# V26 Alpaca Long-Only Production Notebook

This notebook validates the packaged V26 long-only Alpaca paper-trading strategy and runs a dry-run or paper-order pass.

In [ ]:
# Install dependencies
!pip -q install -r requirements.txt

In [ ]:
# Optional: create .env manually or set env vars in Colab secrets
from pathlib import Path
print(Path('.env.template').read_text())

In [ ]:
from src.v26_crypto_trader.settings import load_config
from src.v26_crypto_trader.alpaca_data import load_crypto_bars_alpaca, bars_to_close_volume
from src.v26_crypto_trader.strategy import generate_target_weights
from src.v26_crypto_trader.risk import apply_risk_limits

cfg = load_config('config/frozen_config.yaml')
print(cfg['strategy'])

In [ ]:
# Generate latest long-only target weights without placing orders
bars = load_crypto_bars_alpaca(cfg['universe']['symbols'], lookback_days=cfg['universe']['lookback_days'])
close, volume = bars_to_close_volume(bars)
signal = generate_target_weights(close, cfg)
weights = apply_risk_limits(signal.target_weights, cfg)
print('asof:', signal.asof_date)
print('active:', signal.active)
print('reason:', signal.reason)
print('regime:', signal.regime_snapshot)
weights[weights > 0].to_frame('target_weight')

In [ ]:
# Plan paper orders. Set EXECUTE_PAPER_ORDERS=true only after reviewing dry-run output.
from src.v26_crypto_trader.run_daily import main
signal_df, order_df, positions, perf_df = main('config/frozen_config.yaml')
print('Signals')
display(signal_df)
print('Orders')
display(order_df)
print('Positions')
display(positions)
print('Performance snapshot')
display(perf_df)